# TIFF 标签自动重映射：多数像素 -> 255，少数像素 -> 0

用途：把分割 TIFF 统一转换为项目常用标签约定：`255 = solid`，`0 = pore / void`。

使用方法：只需要修改下一格里的 `input_path`，然后从上到下运行全部单元格。

默认设置会覆盖源文件。覆盖时会先写临时 TIFF，验证成功后再替换源文件，避免写到一半损坏原文件。输出默认不压缩，兼容性最好。

In [ ]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import tifffile as tif

# 只需要改这里：填入要处理的 TIFF 文件路径。
input_path = r"C:\Users\imgw\Desktop\5-89-seged.tiff"

# 默认覆盖源文件，满足“只输入文件路径即可”的批处理习惯。
# 如需保留源文件，改成 False，并在 output_path 中写入新文件路径。
overwrite_source = True
output_path = None  # 例如 r"C:\Users\imgw\Desktop\converted_solid255_pore0.tiff"

# 输出 TIFF 压缩方式。None 表示不压缩，兼容性最好。
# 不建议改成 "zlib"，部分 Anaconda/tifffile 旧环境读取 Deflate TIFF 会报错。
compression = None

In [ ]:
def read_tiff_array(path):
    """Read a TIFF stack, falling back to Pillow for old tifffile/Deflate issues."""
    path = Path(path)
    try:
        return tif.imread(path), "tifffile"
    except Exception as first_error:
        try:
            from PIL import Image
        except Exception as pillow_error:
            raise RuntimeError(
                "tifffile 读取失败，且当前环境没有可用的 Pillow 备用读取器。"
            ) from pillow_error

        try:
            with Image.open(path) as img:
                n_frames = int(getattr(img, "n_frames", 1))
                img.seek(0)
                first = np.asarray(img).copy()
                if n_frames == 1:
                    return first, "pillow_fallback_after_tifffile_error"

                stack = np.empty((n_frames, *first.shape), dtype=first.dtype)
                stack[0] = first
                for frame_index in range(1, n_frames):
                    img.seek(frame_index)
                    stack[frame_index] = np.asarray(img)
                return stack, "pillow_fallback_after_tifffile_error"
        except Exception as pillow_read_error:
            raise RuntimeError(
                f"TIFF 读取失败。tifffile 原始错误: {type(first_error).__name__}: {first_error}; "
                f"Pillow 备用读取也失败: {type(pillow_read_error).__name__}: {pillow_read_error}"
            ) from pillow_read_error


def iter_count_chunks(arr):
    """Yield small array chunks so counting never casts the whole 3D stack at once."""
    if arr.ndim >= 3:
        for index in range(arr.shape[0]):
            yield arr[index]
    else:
        yield arr


def count_integer_labels(arr):
    """Count integer labels without sorting/copying the full TIFF stack."""
    if np.issubdtype(arr.dtype, np.integer):
        min_value = int(arr.min())
        max_value = int(arr.max())
        if min_value >= 0 and max_value <= 65535:
            counts_all = np.zeros(max_value + 1, dtype=np.int64)
            for chunk in iter_count_chunks(arr):
                counts_all += np.bincount(chunk.reshape(-1), minlength=max_value + 1)
            values = np.nonzero(counts_all)[0]
            counts = counts_all[values]
            return values.astype(arr.dtype, copy=False), counts

    return np.unique(arr, return_counts=True)


def remap_tiff_majority_to_solid255(
    input_path,
    *,
    overwrite_source=True,
    output_path=None,
    compression=None,
):
    """Remap the most frequent TIFF label to 255 and all other labels to 0."""
    src = Path(input_path).expanduser()
    if not src.exists():
        raise FileNotFoundError(f"找不到输入文件: {src}")
    if src.suffix.lower() not in {".tif", ".tiff"}:
        raise ValueError(f"输入文件不是 TIFF: {src}")

    if overwrite_source:
        dst = src
    else:
        if output_path is None:
            dst = src.with_name(f"{src.stem}_solid255_pore0{src.suffix}")
        else:
            dst = Path(output_path).expanduser()
            dst.parent.mkdir(parents=True, exist_ok=True)

    arr, reader = read_tiff_array(src)
    arr_shape = list(arr.shape)
    arr_size = int(arr.size)
    arr_dtype = str(arr.dtype)
    values, counts = count_integer_labels(arr)
    if len(values) < 2:
        raise ValueError(
            f"至少需要 2 个像素标签才能判断多数/少数；当前只有 {len(values)} 个: {values.tolist()}"
        )

    majority_index = int(np.argmax(counts))
    majority_value = values[majority_index]

    # 低内存转换：避免 np.where 先生成 int32 大数组。
    majority_mask = arr == majority_value
    if arr.dtype == np.uint8:
        converted = arr
        converted[...] = 0
    else:
        converted = np.zeros(arr.shape, dtype=np.uint8)
    converted[majority_mask] = 255
    del majority_mask

    write_target = dst
    if overwrite_source:
        write_target = src.with_name(f"{src.stem}.tmp_solid255_pore0{src.suffix}")

    tif.imwrite(write_target, converted, photometric="minisblack", compression=compression)
    del arr
    del converted

    check, check_reader = read_tiff_array(write_target)
    check_values, check_counts = count_integer_labels(check)
    check_size = int(check.size)
    check_dtype = str(check.dtype)
    expected_values = np.array([0, 255], dtype=np.uint8)
    if not np.array_equal(check_values, expected_values):
        raise RuntimeError(f"写出后标签异常: {check_values.tolist()}")

    if overwrite_source:
        write_target.replace(dst)

    source_counts = {
        str(int(v)): {"count": int(c), "fraction": float(c / arr_size)}
        for v, c in zip(values, counts)
    }
    output_counts = {
        str(int(v)): {"count": int(c), "fraction": float(c / check_size)}
        for v, c in zip(check_values, check_counts)
    }
    metadata = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_path": str(src),
        "output_path": str(dst),
        "overwrote_source": bool(dst == src),
        "shape": arr_shape,
        "source_dtype": arr_dtype,
        "output_dtype": check_dtype,
        "input_reader": reader,
        "verification_reader": check_reader,
        "source_label_counts": source_counts,
        "mapping_rule": "most frequent source label -> 255; all other source labels -> 0",
        "majority_source_label_mapped_to_255": int(majority_value),
        "minority_or_other_source_labels_mapped_to_0": [
            int(v) for v in values.tolist() if v != majority_value
        ],
        "output_label_counts": output_counts,
    }
    return metadata


metadata = remap_tiff_majority_to_solid255(
    input_path,
    overwrite_source=overwrite_source,
    output_path=output_path,
    compression=compression,
)

print(json.dumps(metadata, ensure_ascii=False, indent=2))